In [64]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

In [2]:
pwd()

'/project/imoskowitz/yubin'

In [3]:
adata = sc.read_h5ad("1-sc_practice/Data/SmoNull_Brain_system.h5ad")

In [4]:
np.isnan(adata.X).sum()

0

In [65]:
adata

AnnData object with n_obs × n_vars = 38311 × 33696
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Condition', 'Library.ident', 'Sample', 'Replicate', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.1', 'seurat_clusters', 'doublet_finder', 'doublet_status', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'SCT_snn_res.1', 'SCT_snn_res.4', 'Extended_mouse_gastrulation_label', 'System', 'ClusterSystem', 'total_counts', 'size_factors', 'Celltype', 'leiden_0_25_log1p', 'leiden_0_25_scran', 'leiden_0_25_pearson', 'leiden_0_5_log1p', 'leiden_0_5_scran', 'leiden_0_5_pearson', 'leiden_1_log1p', 'leiden_1_scran', 'leiden_1_pearson', 'leiden_2_log1p', 'leiden_2_scran', 'leiden_2_pearson', 'leiden_3_log1p', 'leiden_3_scran', 'leiden_3_pearson', 'leiden_5_log1p', 'leiden_5_scran', 'leiden_5_pearson'
    var: 'features', 'highly_variable_log1p', 'highly_variable_nbatches_log1p', 'highly_variable_intersection_log1p', 'mean', 'std', 'means', 'variances', 'residual_variances', 'highly_va

In [78]:
# Convert X to pearson
adata.X = adata.layers["analytic_pearson_residuals"]

In [79]:
df = adata.to_df()

In [80]:
np.isnan(df).sum().sum()

0

In [81]:
(np.isnan(df).all(axis=0) == True).sum()

0

In [85]:
(df == 0).all(axis=0).sum()

5046

In [82]:
adata.X = adata.layers["scran_normalization"]

In [83]:
df_scran = adata.to_df()

In [84]:
(df_scran==0).all(axis=0).sum()

5046

In [45]:
adata.X = adata.layers["log1p_norm"]

In [46]:
df_log = adata.to_df()

In [58]:
(df_log==0).all(axis=0).sum()

5046

## Conclusion
It appears all the NaN present in pearson are in columns where all value are 0. This is because pearson residual calculates the resdiual per gene (col), so when the col contain only 0, it essentially is dividing by 0 (since calculation of denominator is the variance: expected + expected^2/theta). Division by 0 returns NaN

In [62]:
# Will convert all NaN values to 0, matching what was seen in the log and scran normalized dataframe

In [70]:
import scipy.sparse as sp
if sp.issparse(adata.X):
    adata.X = adata.X.toarray()
adata.X[np.isnan(adata.X)] = 0

In [76]:
adata.X = sp.csr_matrix(adata.X)

In [77]:
adata.layers["analytic_pearson_residuals"] = adata.X

In [86]:
# Saving the adata object with the updated pearson residuals layer
adata.write_h5ad("1-sc_practice/Data/SmoNull_Brain_system.h5ad")